# Summary, Claims, and Citations

This notebook takes one clinical note, generates a structured summary with `ClinicalSummarizer`, decomposes that summary into atomic claims, and shows citation snippets and line numbers for each supported claim.

- Set `NOTE_TEXT` to paste a note directly, or point `NOTE_PATH` at a local file.
- Runtime note: the summary and claim-verification cells call the local MLX Qwen model, so they can take a while to run.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "clinical_summarization.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the LLMS repo root from the current notebook working directory."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from clinical_summarization import ClinicalSummarizer, ClinicalSummarizerConfig
from prompts import CLINICAL_SUMMARY_END_MARKER

pd.set_option("display.max_colwidth", 200)


def strip_terminal_marker(text: str) -> str:
    stripped = (text or "").rstrip()
    if stripped.endswith(CLINICAL_SUMMARY_END_MARKER):
        return stripped[: -len(CLINICAL_SUMMARY_END_MARKER)].rstrip()
    return stripped


def load_note_input(note_text: str, note_path: Path | None) -> tuple[str, str]:
    if note_text.strip():
        return note_text.strip(), "inline text"
    if note_path is None:
        raise ValueError("Set NOTE_TEXT or NOTE_PATH before running the notebook.")
    resolved = note_path.expanduser().resolve()
    return resolved.read_text(encoding="utf-8"), str(resolved)


def add_line_numbers(text: str) -> str:
    lines = text.splitlines()
    width = max(3, len(str(len(lines))))
    return "\n".join(
        f"{idx:>{width}} | {line}" for idx, line in enumerate(lines, start=1)
    )


def citation_line_span(citation) -> str:
    if citation.start_line == citation.end_line:
        return str(citation.start_line)
    return f"{citation.start_line}-{citation.end_line}"


def claim_results_to_frame(claim_results) -> pd.DataFrame:
    rows = []
    for index, result in enumerate(claim_results, start=1):
        rows.append(
            {
                "claim_index": index,
                "claim": result.claim,
                "supported": result.supported,
                "confidence": round(result.probability, 3),
                "citation_lines": ", ".join(
                    citation_line_span(citation) for citation in result.citations
                ),
                "n_citations": len(result.citations),
                "top_citation": result.citations[0].snippet if result.citations else "",
            }
        )
    return pd.DataFrame(rows)


def claim_results_to_payload(claim_results) -> list[dict]:
    payload = []
    for index, result in enumerate(claim_results, start=1):
        payload.append(
            {
                "claim_index": index,
                "claim": result.claim,
                "supported": result.supported,
                "confidence": result.probability,
                "citations": [
                    {
                        "snippet": citation.snippet,
                        "start_line": citation.start_line,
                        "end_line": citation.end_line,
                        "start_char": citation.start_char,
                        "end_char": citation.end_char,
                        "score": citation.score,
                        "method": citation.method,
                    }
                    for citation in result.citations
                ],
            }
        )
    return payload


def display_claim_citations(claim_results) -> None:
    for index, result in enumerate(claim_results, start=1):
        display(
            Markdown(
                "\n".join(
                    [
                        f"### Claim {index}",
                        f"**Claim:** {result.claim}",
                        f"**Supported:** {result.supported}",
                        f"**Confidence:** {result.probability:.3f}",
                    ]
                )
            )
        )
        if not result.citations:
            display(Markdown("_No citation snippet attached for this claim._"))
            continue
        for citation_index, citation in enumerate(result.citations, start=1):
            display(
                Markdown(
                    "\n".join(
                        [
                            f"**Citation {citation_index}**",
                            f"- Lines: {citation_line_span(citation)}",
                            f"- Score: {citation.score:.3f}",
                            "",
                            "```text",
                            citation.snippet,
                            "```",
                        ]
                    )
                )
            )


In [2]:
# If NOTE_TEXT is non-empty, it overrides NOTE_PATH.
NOTE_TEXT = ""
NOTE_PATH = REPO_ROOT / "sample_notes" / "synthetic_discharge_summary.txt"

NOTE_TYPE = "discharge summary"
PATIENT_ID = "demo-note-001"
SHOW_SOURCE_NOTE = True

MODEL_PATH = "mlx-community/Qwen3.5-9B-4bit"
TARGET_COMPRESSION_RATIO = 0.18
MAX_TOKENS = 1536
CLAIM_SUPPORT_THRESHOLD = 0.5

ARTIFACT_PATH = REPO_ROOT / "notebooks" / "artifacts" / "summary_claims_citations_demo.json"


In [3]:
note_text, note_source = load_note_input(NOTE_TEXT, NOTE_PATH)
print(f"Loaded note from {note_source}.")
print(f"Source note length: {len(note_text.splitlines())} lines / {len(note_text.split())} words")

if SHOW_SOURCE_NOTE:
    display(Markdown("## Source Note"))
    display(Markdown(f"```text\n{add_line_numbers(note_text)}\n```"))
else:
    print("SHOW_SOURCE_NOTE is False, skipping full source-note display.")


Loaded note from /Users/natejly/Documents/GitHub/LLMS/sample_notes/synthetic_discharge_summary.txt.
Source note length: 1 lines / 593 words


## Source Note

```text
  1 | 63 year old woman with lupus, obesity, prior thyroid lymphoma with    resultant hypothyroidism who was found down at her home- noted to be    bradycardic and hypotensive by EMS, unable to transcutaneously pace in    the field.  In the ED at [**Hospital3 3191**] Hospital she continued to be    bradycardic, in complete heart block by report, she was coded for    bradycardia HR 20s rec'd epi/ atropine/ glucagon/ etc.  A temporary    transvenous pacing wire was placed.  She was intubated, started on    dopamine and propofol and transferred here for further care.  She is in    a hard collar r/t unwitnessed fall, CT head at OSH pnd.  ECHO at OSH    PND. Unclear etiology of CHB.    Heart block, complete (CHB)    Assessment:    R SC transvenous pacing wire remains in place. No ectopy noted. No    pacing noted. HR 80 s-100s NSR/ST.    Action:    *Transvenous pacer tested per [**Hospital1 1**] protocol- see Metavision for    settings. Magnesium/ Phosphate repleted. EKG obtained.    *Permanent PCM on hold given fevers overnight per EP    Response:    Remains in NSR/ST without bradycardia or rhythm disturbance noted.    Plan:    *Permanent PCM once afebrile. Keep temporary wire in place until    permanent PCM. Monitor per hosp policy.    Hypotension (not Shock)    Assessment:    Received Pt. on IV Dopamine at 12mcg/kg/min with BP ranges initially    90-100 s/40-50. CVP 10.  UOP ~50ml/hr, increasing after doses of IV    anbx.  Weak palp radial pulses and weak palp DP/PT.Skin cool/ clammy.    Action:    *Completed 2^nd liter of IVF per order.    *Unclear etiology of hypotension- sepsis vs cardiogenic. PA line placed    over wire (R IJ TLC) under fluoroscopy at 17:00. CVP 10. PCWP 22,    correlating w/ PAD 22.  MVO2 68. HGB/ CO/CI numbers PND.    *Weaned dopamine as able    *ECHO complete->    *Minimizing hypotensive effect of propofol by adding midazolam. (See    below for pain control).    *Pt s/p multiple failed attempts at arterial line [**4-10**]. Following NIBPs.    Response:    *Pt w/ BP 90s-100s, MAPs high 50s-60s on dopamine @ 8mcg/kg/min.  PADs    low 20s.    Plan:    Awaiting team plans given new PA line data. Awaiting CO/CI.   Continue    to wean dopamine as able. Follow fluid/ volume status/ UOP. Monitor for    s/s sepsis/ temp spike.    Fever (Hyperthermia, Pyrexia, not Fever of Unknown Origin)    Assessment:    Tmax-> 100.4 blood. WBC up slightly 11.2 (10.9). Pan cultured overnight    last night.  Pt diaphoretic/ clammy. Lactate 2.0. VS as above.    Action:    *Triple IV anbx- cefepime/ vanco/ flagyl.    *Monitored temp/ WBC    *Tylenol PRN  had been given via NGT, however, later found that    gastric meds not being absorbed.    *Cool baths/ skin care PRN for diaphoresis    Response:    T range 99.8-100.4 Rectal/ blood.    Plan:    Cont to monitor temp (now w/ blood temp monitor on PA line), follow up    with cultures., Antibxs as ordered    Respiratory failure, acute (not ARDS/[**Doctor Last Name 2**])    Assessment:    *Rec d pt on AC FiO2 50%, 500 x 20, 5 peep.    *LS clear apices/ rhonchi throughout. SPO2 >94%. Thick, tan secretions    via ETT. Sedated per below.    Action:    *Likely aspiration pneumonitis from brady event per team, anbx as above    *Weaned vent settings: PS trial 12PS/ 5 peep-> RR up to 40s, pt    appearing uncomfortable in bed.  PS increased to 16/ 5 peep. VENOUS    blood gas correlating to ABG on AC settings per resp. Stayed on these    settings most of the day. Placed on AC, fiO2 100% for PA line    placement.    Response:    *After line placed, FiO2 decreased to 50%. VBG pnd.    Plan:    Cont to monitor resp status, CXR in am, cont IV Antibxs as ordered.    Follow VBGs until arterial line able to be placed.    Pain control (acute pain, chronic pain)    Assessment:    Action:    Response:    Plan:    Cont to monitor pain, may need pain consult to aid in best    sedation/pain medications, cont to monitor neuro status, comfort and    emotional support to Pt. and husband
```

In [4]:
config = ClinicalSummarizerConfig(
    model_path=MODEL_PATH,
    max_tokens=MAX_TOKENS,
    target_compression_ratio=TARGET_COMPRESSION_RATIO,
    claim_support_threshold=CLAIM_SUPPORT_THRESHOLD,
)
summarizer = ClinicalSummarizer(config)
summarizer


In [5]:
summary = strip_terminal_marker(
    summarizer.summarize_text(
        note_text,
        patient_id=PATIENT_ID,
        note_type=NOTE_TYPE,
        source=note_source,
    )
)

display(Markdown("## Structured Summary"))
display(Markdown(f"```text\n{summary}\n```"))


/Users/natejly/Documents/GitHub/LLMS/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 159094.29it/s]


## Structured Summary

```text
Brief Hospital Course:
63F with SLE, obesity, hypothyroidism found down, bradycardic, hypotensive.
ED code for bradycardia HR 20s; failed transcutaneous pacing.
Received epi/atropine/glucagon; placed temporary transvenous pacer.
Intubated, started dopamine/propofol; transferred here.
Hard collar r/t unwitnessed fall; CT head/ECHO pnd.
Diagnoses:
Complete heart block (CHB); Hypotension (not shock); Fever; Acute resp failure.
Procedures / interventions:
R SC transvenous pacing wire placed.
PA line placed R IJ TLC under fluoroscopy.
Medication changes:
Started IV dopamine 12mcg/kg/min; stopped propofol (add midazolam).
New IV anbx: cefepime/vanco/flagyl.
Significant labs / imaging:
Tmax 100.4 blood; WBC 11.2; Lactate 2.0.
Echo: PAD 22, PCWP 22, MVO2 68.
Hospital course by problem:
CHB: Remains NSR/ST 80-100s; pacer tested; permanent PCM on hold.
Hypotension: Dopamine 8mcg/kg/min; BP 90s-100s; MAP 50s-60s; weaning.
Fever: T 99.8-100.4; WBC up; triple IV anbx; temp monitor on PA.
Resp failure: AC 50% FiO2; weaned to 100% for PA; thick tan secretions.
Discharge disposition & follow-up:
Permanent PCM once afebrile; keep temp wire; monitor resp/volume.
Follow cultures; continue IV anbx; monitor pain/neuro status.
```

In [6]:
verification_result = summarizer.verify_generated_summary(
    summary,
    source_note_text=note_text,
    patient_id=PATIENT_ID,
)

print(
    f"Generated {len(verification_result.claims)} claims; unsupported claims: {len(verification_result.unsupported_claims)}"
)

claims_df = claim_results_to_frame(verification_result.claim_results)
display(Markdown("## Claim Decomposition"))
display(claims_df)


Generated 47 claims; unsupported claims: 11


## Claim Decomposition

,claim_index,claim,supported,confidence,citation_lines,n_citations,top_citation
0,1,"Patient 63F with SLE, obesity, hypothyroidism.",True,1.0,"1, 1",2,"63 year old woman with lupus, obesity, prior thyroid lymphoma with resultant hypothyroidism who was found down at her home- noted to be bradycardic and hypotensive by EMS, unable to transcut..."
1,2,"Patient found down, bradycardic, hypotensive.",True,1.0,"1, 1",2,"63 year old woman with lupus, obesity, prior thyroid lymphoma with resultant hypothyroidism who was found down at her home- noted to be bradycardic and hypotensive by EMS, unable to transcut..."
2,3,ED code for bradycardia HR 20s.,True,1.0,"1, 1",2,"In the ED at [**Hospital3 3191**] Hospital she continued to be bradycardic, in complete heart block by report, she was coded for bradycardia HR 20s rec'd epi/ atropine/ glucagon/ etc."
3,4,Transcutaneous pacing failed.,True,1.0,"1, 1",2,"63 year old woman with lupus, obesity, prior thyroid lymphoma with resultant hypothyroidism who was found down at her home- noted to be bradycardic and hypotensive by EMS, unable to transcut..."
4,5,Received epi/atropine/glucagon.,True,1.0,"1, 1",2,Hypotension (not Shock) Assessment: Received Pt.
5,6,Temporary transvenous pacer placed.,True,1.0,"1, 1",2,A temporary transvenous pacing wire was placed.
6,7,Patient intubated.,True,1.0,"1, 1",2,"She was intubated, started on dopamine and propofol and transferred here for further care."
7,8,Started IV dopamine/propofol.,True,1.0,1,1,"63 year old woman with lupus, obesity, prior thyroid lymphoma with resultant hypothyroidism who was found down at her home- noted to be bradycardic and hypotensive by EMS, unable to transcut..."
8,9,Patient transferred here.,True,1.0,"1, 1",2,"She was intubated, started on dopamine and propofol and transferred here for further care."
9,10,Hard collar placed r/t unwitnessed fall.,True,1.0,"1, 1",2,"She is in a hard collar r/t unwitnessed fall, CT head at OSH pnd."


In [7]:
display(Markdown("## Claim-Level Citations"))
display_claim_citations(verification_result.claim_results)


## Claim-Level Citations

### Claim 1
**Claim:** Patient 63F with SLE, obesity, hypothyroidism.
**Supported:** True
**Confidence:** 1.000

**Citation 1**
- Lines: 1
- Score: 0.376

```text
63 year old woman with lupus, obesity, prior thyroid lymphoma with    resultant hypothyroidism who was found down at her home- noted to be    bradycardic and hypotensive by EMS, unable to transcutaneously pace in    the field.
```

**Citation 2**
- Lines: 1
- Score: 0.352

```text
63 year old woman with lupus, obesity, prior thyroid lymphoma with    resultant hypothyroidism who was found down at her home- noted to be    bradycardic and hypotensive by EMS, unable to transcutaneously pace in    the field.  In the ED at [**Hospital3 3191**] Hospital she continued to be    bradycardic, in complete heart block by report, she was coded for    bradycardia HR 20s rec'd epi/ atropine/ glucagon/ etc.  A temporary    transvenous pacing wire was placed.  She was intubated, started on    dopamine and propofol and transferred here for further care.  She is in    a hard collar r/t unwitnessed fall, CT head at OSH pnd.  ECHO at OSH    PND. Unclear etiology of CHB.    Heart block, complete (CHB)    Assessment:    R SC transvenous pacing wire remains in place. No ectopy noted. No    pacing noted. HR 80 s-100s NSR/ST.    Action:    *Transvenous pacer tested per [**Hospital1 1**] protocol- see Metavision for    settings. Magnesium/ Phosphate repleted. EKG obtained.    *Permanent PCM on hold given fevers overnight per EP    Response:    Remains in NSR/ST without bradycardia or rhythm disturbance noted.    Plan:    *Permanent PCM once afebrile. Keep temporary wire in place until    permanent PCM. Monitor per hosp policy.    Hypotension (not Shock)    Assessment:    Received Pt. on IV Dopamine at 12mcg/kg/min with BP ranges initially    90-100 s/40-50. CVP 10.  UOP ~50ml/hr, increasing after doses of IV    anbx.  Weak palp radial pulses and weak palp DP/PT.Skin cool/ clammy.    Action:    *Completed 2^nd liter of IVF per order.    *Unclear etiology of hypotension- sepsis vs cardiogenic. PA line placed    over wire (R IJ TLC) under fluoroscopy at 17:00. CVP 10. PCWP 22,    correlating w/ PAD 22.  MVO2 68. HGB/ CO/CI numbers PND.    *Weaned dopamine as able    *ECHO complete->    *Minimizing hypotensive effect of propofol by adding midazolam. (See    below for pain control).    *Pt s/p multiple failed attempts at arterial line [**4-10**]. Following NIBPs.    Response:    *Pt w/ BP 90s-100s, MAPs high 50s-60s on dopamine @ 8mcg/kg/min.  PADs    low 20s.    Plan:    Awaiting team plans given new PA line data. Awaiting CO/CI.   Continue    to wean dopamine as able. Follow fluid/ volume status/ UOP. Monitor for    s/s sepsis/ temp spike.    Fever (Hyperthermia, Pyrexia, not Fever of Unknown Origin)    Assessment:    Tmax-> 100.4 blood. WBC up slightly 11.2 (10.9). Pan cultured overnight    last night.  Pt diaphoretic/ clammy. Lactate 2.0. VS as above.    Action:    *Triple IV anbx- cefepime/ vanco/ flagyl.    *Monitored temp/ WBC    *Tylenol PRN  had been given via NGT, however, later found that    gastric meds not being absorbed.    *Cool baths/ skin care PRN for diaphoresis    Response:    T range 99.8-100.4 Rectal/ blood.    Plan:    Cont to monitor temp (now w/ blood temp monitor on PA line), follow up    with cultures., Antibxs as ordered    Respiratory failure, acute (not ARDS/[**Doctor Last Name 2**])    Assessment:    *Rec d pt on AC FiO2 50%, 500 x 20, 5 peep.    *LS clear apices/ rhonchi throughout. SPO2 >94%. Thick, tan secretions    via ETT. Sedated per below.    Action:    *Likely aspiration pneumonitis from brady event per team, anbx as above    *Weaned vent settings: PS trial 12PS/ 5 peep-> RR up to 40s, pt    appearing uncomfortable in bed.  PS increased to 16/ 5 peep. VENOUS    blood gas correlating to ABG on AC settings per resp. Stayed on these    settings most of the day. Placed on AC, fiO2 100% for PA line    placement.    Response:    *After line placed, FiO2 decreased to 50%. VBG pnd.    Plan:    Cont to monitor resp status, CXR in am, cont IV Antibxs as ordered.    Follow VBGs until arterial line able to be placed.    Pain control (acute pain, chronic pain)    Assessment:    Action:    Response:    Plan:    Cont to monitor pain, may need pain consult to aid in best    sedation/pain medications, cont to monitor neuro status, comfort and    emotional support to Pt. and husband
```

### Claim 2
**Claim:** Patient found down, bradycardic, hypotensive.
**Supported:** True
**Confidence:** 1.000

...

_No citation snippet attached for this claim._

### Claim 46
**Claim:** Continue IV anbx.
**Supported:** False
**Confidence:** 0.000

_No citation snippet attached for this claim._

### Claim 47
**Claim:** Monitor pain/neuro status.
**Supported:** False
**Confidence:** 0.000

_No citation snippet attached for this claim._

In [8]:
artifact_payload = {
    "note_source": note_source,
    "patient_id": PATIENT_ID,
    "note_type": NOTE_TYPE,
    "summary": summary,
    "unsupported_claims": list(verification_result.unsupported_claims),
    "claims": claim_results_to_payload(verification_result.claim_results),
}

ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH.write_text(json.dumps(artifact_payload, indent=2), encoding="utf-8")
print(f"Wrote notebook artifact to {ARTIFACT_PATH}")

artifact_payload["claims"][0] if artifact_payload["claims"] else artifact_payload


Wrote notebook artifact to /Users/natejly/Documents/GitHub/LLMS/notebooks/artifacts/summary_claims_citations_demo.json


{'claim_index': 1,
 'claim': 'Patient 63F with SLE, obesity, hypothyroidism.',
 'supported': True,
 'confidence': 1.0,
 'citations': [{'snippet': '63 year old woman with lupus, obesity, prior thyroid lymphoma with    resultant hypothyroidism who was found down at her home- noted to be    bradycardic and hypotensive by EMS, unable to transcutaneously pace in    the field.',
   'start_line': 1,
   'end_line': 1,
   'start_char': 0,
   'end_char': 226,
   'score': 0.3760869565217391,
   'method': 'token_overlap'},
  {'snippet': "63 year old woman with lupus, obesity, prior thyroid lymphoma with    resultant hypothyroidism who was found down at her home- noted to be    bradycardic and hypotensive by EMS, unable to transcutaneously pace in    the field.  In the ED at [**Hospital3 3191**] Hospital she continued to be    bradycardic, in complete heart block by report, she was coded for    bradycardia HR 20s rec'd epi/ atropine/ glucagon/ etc.  A temporary    transvenous pacing wire was placed